# Antenna Basics

This notebook introduces antenna patterns, gain, and polarization by visualizing simple directional models rather than jumping straight into hardware details.

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd().resolve().parent
if str(ROOT) not in sys.path:
    sys.path.append(str(ROOT))

from rf_utils import *
from IPython.display import Audio, Markdown, display
import ipywidgets as widgets
import matplotlib.pyplot as plt
import numpy as np
from scipy import signal

%matplotlib widget

plt.rcParams.update({
    "figure.figsize": (12, 4),
    "axes.grid": True,
    "grid.alpha": 0.3,
    "font.size": 11,
})


## Radiation Patterns

Antenna gain does not create power from nowhere. It redistributes where the power goes.

In [ ]:
theta = np.linspace(0, 2 * np.pi, 1000)
patterns = {
    "Isotropic": np.ones_like(theta),
    "Dipole": np.abs(np.sin(theta)),
    "Ground plane": 0.6 + 0.4 * np.abs(np.sin(theta)),
    "Yagi": np.clip(np.cos(theta / 2), 0, None) ** 6,
}

fig = plt.figure(figsize=(7, 7))
ax = plt.subplot(111, projection="polar")

def update_pattern(pattern_name="Dipole"):
    ax.clear()
    ax.plot(theta, patterns[pattern_name])
    ax.set_title(pattern_name)
    fig.canvas.draw_idle()

controls = widgets.interactive(
    update_pattern,
    pattern_name=dropdown(options=list(patterns.keys()), value="Dipole", description="Pattern"),
)
display(controls)


## SWR Intuition

SWR comes from reflections caused by mismatch. Reflection coefficient magnitude `|Gamma|` maps directly to SWR:

$$SWR = \frac{1 + |\Gamma|}{1 - |\Gamma|}$$

In [ ]:
out = widgets.Output()

def update_swr(gamma=0.2):
    with out:
        out.clear_output(wait=True)
        swr = (1 + gamma) / max(1 - gamma, 1e-9)
        print(f"Reflection coefficient magnitude: {gamma:.2f}")
        print(f"SWR: {swr:.2f}:1")

controls = widgets.interactive(
    update_swr,
    gamma=float_slider(min_value=0, max_value=0.95, step=0.01, value=0.2, description="|Gamma|"),
)
display(controls, out)


## Key Takeaway

Antennas are spatial filters. Pattern and matching determine where power goes and how efficiently it moves between the transmitter, feedline, and free space.